# Step 9 - Neutral Rewriting Module: rule-based, fine-tuned seq2seq and LLM-guided rewriting with a semantic-preservation gate

**Project:** EQUAL - gender bias in Philippine PESO-linked job advertisements (`equal-data-set.xlsx`, 1,500 advertisements).
**Task:** for each unit that Step 5b flagged as explicit (E1, E2, E3) or implicit (I1, I2), propose a neutral rewrite that keeps the job requirement, and decide automatically whether the rewrite is safe to accept.
**Scope:** the same 400-advertisement test subset as Steps 7 and 8; the 1,100 other advertisements are the training pool.

### What this notebook does and does not do
- It builds the three approaches described in the brief and a cascade that tries them in order: **Approach 1** rule-based substitution, **Approach 2** fine-tuned seq2seq (`t5-small`), **Approach 3** LLM-guided rewriting.
- **The LLM is `Qwen/Qwen2.5-1.5B-Instruct`, run locally on CPU, not GPT-4.** GPT-4 is not reachable from this environment, and sending corpus text to a third party is not permitted. Every result labelled "Approach 3" is a result for this small model and says nothing about GPT-4.
- **Evaluation is automatic only.** No human rated any rewrite, and no rating was invented. The four human-rated parameters in the brief (readability, neutrality/perceived bias, content preservation, grammaticality/fluency, 7-point Likert) are not measured here; the automatic checks below are proxies for them and are named as proxies.
- **No human annotation and no mock data.** All inputs are the preprocessed PESO units and the Step 5b rule-derived (silver) spans. The brief's four example phrases are used only as unit tests of the rule templates, and are labelled as such.
- Every "does the rewrite still contain bias?" result uses the **same word lists that produced the silver labels** (Steps 5, 7 and 8). It tests consistency with the rules, not validity; it cannot show that a human reader would find the rewrite neutral.

### Literature base (2022-2025 unless stated)
Verified in `literature-support-for-methods.md`; reading depth is stated where it matters. Hu *et al.* [1] propose job-ad rewording that stays close to the original; He and Kang [2] report that replacing masculine-coded wording changed applicant-pool diversity; Amrhein *et al.* [3] and Mainardi *et al.* [4] study seq2seq and prompted rewriting for gender-fair text (German and Italian, not job ads); Shu *et al.* [5] study instruction-controlled rewriting; Gallegos *et al.* [6] and Davis *et al.* [7] support prompting without retraining for debiasing and for minimal edits; Khondaker *et al.* [8] support a generate-then-verify pipeline that flags what it cannot safely edit; Kumar *et al.* [9] use a lexicon-based neutrality score for job ads; Blackwell *et al.* [10] report that LLM output can vary at fixed settings.

**Gaps stated up front.** We found no verified 2022-2025 source for (a) a cosine threshold for rewrite acceptance, (b) a content-word retention threshold, (c) training a seq2seq model on rule-derived pairs, (d) Flesch readability on job-ad fragments, or (e) bootstrap intervals over advertisements. These are design choices of this study and are reported as such. The brief's references [8], [10] and [xa]-[xd] are placeholders that cannot be resolved from this workspace, so they are not cited.

In [1]:
import os
os.environ["HF_HUB_OFFLINE"] = "1"
import json, re, time, random, math, difflib
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 230)

SMOKE = os.environ.get("S9_SMOKE") == "1"
OUT = Path("outputs") / ("s9_neutral_rewriting_smoke" if SMOKE else "s9_neutral_rewriting")
OUT.mkdir(parents=True, exist_ok=True)
S7 = Path("outputs") / "s7_explicit_detection"

PARAMS = {
    "test_sample": "the 400 ads of Step 7 (read from outputs/s7_explicit_detection/test_sample_400_ads.csv)",
    "val_share_of_pool": 0.10, "val_seed": 42,
    "targets": "units with at least one valid-offset Step 5b span coded E1/E2/E3 (explicit) or I1/I2 (implicit)",
    "not_rewritten": "explicit units that list both genders (G1-style listing, e.g. 'Male and Female') are left unchanged and counted separately",
    "gate": {"embedding_model": "paraphrase-multilingual-MiniLM-L12-v2", "cosine_min": 0.70, "content_retention_min": 0.80,
             "numbers": "digits in the rewrite must equal digits in the original", "length_ratio_range": [0.4, 2.0],
             "cosine_and_length_apply_when": "always, except for a unit whose only flagged wording is explicit and which keeps no content words after that wording is removed (a pure gender restriction)"},
    "template_default": "Open to all qualified candidates",
    "t5": {"model": "google-t5/t5-small", "prefix": "neutralize: ", "seeds": [1, 2, 3], "epochs": 20, "batch": 16, "lr": 3e-4, "weight_decay": 0.01,
           "max_source_len": 128, "max_target_len": 128, "num_beams": 4, "identity_pairs": 250, "identity_seed": 42},
    "llm": {"model": "Qwen/Qwen2.5-1.5B-Instruct", "device": "cpu", "decoding": "greedy (do_sample=False)", "max_new_tokens": 80, "examples_in_prompt": 0,
            "constraints": ["preserve all factual job requirements verbatim", "eliminate the identified gender-preferential language", "maintain the original sentence structure"]},
    "cascade_order": ["A1 rules", "A2 t5-small (seed 1)", "A3 Qwen2.5-1.5B"],
    "tau_grid": [0.5, 0.6, 0.7, 0.8, 0.9],
    "bootstrap_resamples": 1000, "bootstrap_seed": 7, "ci": "95% percentile, ads resampled with replacement",
}
if SMOKE:
    PARAMS["t5"].update({"seeds": [1], "epochs": 2})
json.dump(PARAMS, open(OUT / "parameters.json", "w"), indent=1)
print("SMOKE RUN" if SMOKE else "FULL RUN")
print(json.dumps(PARAMS, indent=1)[:2500])

rd = lambda p: pd.read_csv(p, encoding="utf-8-sig")
units = rd("outputs/analysis/units.csv")
ad_ctx = rd("outputs/s5b_taxonomy_test/ad_context_v0.3.csv")
sp_all = rd("outputs/s5b_taxonomy_test/preannotation_spans_v0.3.csv")
ad_ctx["route_v03"] = ad_ctx["route_v03"].fillna("no span")
print(f"{units['record_id'].nunique():,} ads, {len(units):,} units, {len(sp_all):,} v0.3 spans")

FULL RUN
{
 "test_sample": "the 400 ads of Step 7 (read from outputs/s7_explicit_detection/test_sample_400_ads.csv)",
 "val_share_of_pool": 0.1,
 "val_seed": 42,
 "targets": "units with at least one valid-offset Step 5b span coded E1/E2/E3 (explicit) or I1/I2 (implicit)",
 "not_rewritten": "explicit units that list both genders (G1-style listing, e.g. 'Male and Female') are left unchanged and counted separately",
 "gate": {
  "embedding_model": "paraphrase-multilingual-MiniLM-L12-v2",
  "cosine_min": 0.7,
  "content_retention_min": 0.8,
  "numbers": "digits in the rewrite must equal digits in the original",
  "length_ratio_range": [
   0.4,
   2.0
  ],
  "cosine_and_length_apply_when": "always, except for a unit whose only flagged wording is explicit and which keeps no content words after that wording is removed (a pure gender restriction)"
 },
 "template_default": "Open to all qualified candidates",
 "t5": {
  "model": "google-t5/t5-small",
  "prefix": "neutralize: ",
  "seeds": [
   

## 1. Test subset and training pool

The test subset is the Step 7 sample (400 advertisements, seed 20261026) so that Steps 7, 8 and 9 are evaluated on identical advertisements. The remaining 1,100 advertisements are split 990 train / 110 validation by route (seed 42), identical to Steps 7 and 8. Training data for Approach 2 come from the 990 training advertisements only. No test unit enters any training or tuning step.

In [2]:
test_ads = sorted(rd(S7 / "test_sample_400_ads.csv")["record_id"])
route_of = dict(zip(ad_ctx["record_id"], ad_ctx["route_v03"]))
unit_ad = dict(zip(units["unit_id"], units["record_id"]))
text_of = dict(zip(units["unit_id"], units["text_masked"].astype(str)))
comp_of = dict(zip(units["unit_id"], units["component"]))

pool_ads = sorted(set(units["record_id"]) - set(test_ads))
rng_split = np.random.RandomState(PARAMS["val_seed"])
pool_route = {a: route_of[a] for a in pool_ads}
val_ads = set()
for k in sorted(set(pool_route.values())):
    ids = sorted(a for a, r in pool_route.items() if r == k)
    n_val = max(1, int(round(len(ids) * PARAMS["val_share_of_pool"])))
    val_ads |= set(rng_split.choice(ids, n_val, replace=False))
train_ads = sorted(set(pool_ads) - val_ads)
split_of = {**{a: "train" for a in train_ads}, **{a: "val" for a in val_ads}, **{a: "test" for a in test_ads}}
assert not (set(train_ads) | val_ads) & set(test_ads)
print(f"train {len(train_ads)} ads; val {len(val_ads)} ads; test {len(test_ads)} ads")

train 990 ads; val 110 ads; test 400 ads


## 2. Units that need a rewrite

A unit is a **target** when Step 5b gave it at least one valid-offset explicit span (E1 gender restriction or preference, E2 gender-marked occupational wording, E3 generic gendered pronoun) or implicit span (I1 masculine-coded, I2 feminine-coded wording). A target is `explicit`, `implicit` or `both`.

Two groups are handled separately:
- **Listings that name both genders** ("Male and Female", "Male, Female, LGBT") are not restrictions, but Step 5b coded some of them E1, which Step 4b recorded as an unresolved E1/G1 conflict. We leave them unchanged and report them as `no_rewrite`.
- **Role-noun implicit spans** (Step 5 `P_candidate_role_noun`, for example "lead cashier") may not be biased at all. They stay in the target set but are flagged, and a sensitivity analysis drops them.

In [3]:
sp = sp_all[sp_all["span_offset_valid"].astype(str).str.lower().isin(["true", "1"])].copy()
EXPC, IMPC = {"E1", "E2", "E3"}, {"I1", "I2"}
TARGET = {}
for r in sp[sp["expression_code"].isin(EXPC | IMPC)].itertuples():
    t = TARGET.setdefault(r.unit_id, {"exp": [], "imp": [], "imp_flags": []})
    item = (int(r.span_start), int(r.span_end), r.expression_code, r.span_text)
    if r.expression_code in EXPC:
        t["exp"].append(item)
    else:
        t["imp"].append(item); t["imp_flags"].append(r.review_flag if isinstance(r.review_flag, str) else "")
bad = [u for u, t in TARGET.items() for (s, e, c, w) in t["exp"] + t["imp"] if text_of[u][s:e] != w]
print("spans that do not reproduce inside their unit text:", len(bad))

MALE_RX = re.compile(r"\bmales?\b|\bmen\b|\bgentlem[ae]n\b", re.I)
FEM_RX = re.compile(r"\bfemales?\b|\bwom[ae]n\b|\bladies\b|\blady\b", re.I)
COND_RX = re.compile(r"\bfor\s+(?:fe)?male\b", re.I)
is_listing = lambda t: bool(MALE_RX.search(t) and FEM_RX.search(t)) and not COND_RX.search(t)

rows = []
for u, t in TARGET.items():
    kind = "both" if (t["exp"] and t["imp"]) else ("explicit" if t["exp"] else "implicit")
    rows.append({"unit_id": u, "record_id": unit_ad[u], "split": split_of[unit_ad[u]], "component": comp_of[u], "kind": kind,
                 "listing": bool(t["exp"]) and is_listing(text_of[u]),
                 "role_noun_only": bool(t["imp"]) and not t["exp"] and all(f == "P_candidate_role_noun" for f in t["imp_flags"]),
                 "n_exp": len(t["exp"]), "n_imp": len(t["imp"]), "text": text_of[u]})
tg = pd.DataFrame(rows)
print(tg.groupby(["split", "kind"]).size().unstack(fill_value=0))
print("explicit/both units that list both genders (no_rewrite):", tg[tg.listing].groupby("split").size().to_dict())
print("implicit-only units flagged only as role nouns:", tg[tg.role_noun_only].groupby("split").size().to_dict())
tg.drop(columns=["text"]).to_csv(OUT / "target_units.csv", index=False)

spans that do not reproduce inside their unit text: 0
kind   both  explicit  implicit
split                          
test      2        94       153
train     0       233       404
val       0        26        36
explicit/both units that list both genders (no_rewrite): {'train': 1, 'val': 2}
implicit-only units flagged only as role nouns: {'test': 19, 'train': 76, 'val': 2}


## 3. Approach 1 - rule-based substitution (baseline)

Following the brief, Approach 1 applies context-aware templates to the explicit patterns that Step 5b localised. It only handles explicit units; implicit wording has no rule here, because replacing a coded word ("analytical", "support") changes what is being asked and cannot be done by a fixed template.

| Rule | Applies to | Action |
|---|---|---|
| T1 | E1 where the unit is only a gender restriction ("Male", "Preferably Female", "Must be female", "Male only") | replace with "Open to all qualified candidates" |
| T2 | E1 with other content ("Female, college graduate") | delete the gender phrase and its separator; keep the rest |
| T3 | E1 followed by an age range ("Male, 25-35 years old") | "Candidates aged 25-35 years old" |
| T4 | E1 in a title parenthetical ("HR Staff (Preferably Male)") | delete the parenthetical |
| T5 | E2 gender-marked occupation | replace through a fixed table (salesman, saleslady to sales representative; foreman to foreperson; watchman to security guard) |
| T6 | E3 generic pronoun | his/her to "their"; he/she to "they", with verb agreement fixed for is/has/does/was |
| review | gender-conditioned requirements ("height ... for Male and ... for Female"), pregnancy conditions | no rewrite and **no cascade**: sent to manual review (see note below) |
| review | ambiguous E2 ("man", "boy") | no rule rewrite; the cascade tries Approach 2 and 3 |

*Note on the manual-review rule.* In a first full run, Approach 3 rewrote "Female SRs must not be pregnant" as "All qualified candidates must not be pregnant" and the gate accepted it; for the height line it invented a female height. A pregnancy or sex-conditioned physical condition is not wording to neutralise but a requirement to judge, so these units now stop at manual review, as in Step 10. This rule was added after reading test outputs; it changes 2 test units (one accepted, one already rejected).

The E2 replacement table and "Open to all qualified candidates" come from the project brief and Step 10; we found no verified 2022-2025 source that fixes the wording of neutral occupational titles, so the table is a design choice and is listed in `rule_templates.csv`. Keeping the original text and changing only the flagged wording follows the "alternative wording closely related to the original" idea of Hu *et al.* [1].

In [4]:
E2_MAP = {"salesman": "sales representative", "salesmen": "sales representatives", "saleslady": "sales representative", "salesladies": "sales representatives",
          "foreman": "foreperson", "foremen": "forepersons", "watchman": "security guard", "watchmen": "security guards"}
AGREE = {"is": "are", "has": "have", "does": "do", "was": "were"}
PHR = re.compile(r"\b(?:(?:must\s+be|preferably|for)\s+)?(?:fe)?male\b(?:\s+(?:only|preferably|applicants?|candidates?))?", re.I)
PAREN = re.compile(r"\s*\(\s*(?:(?:preferably|must\s+be|for)\s+)?(?:fe)?male(?:\s+(?:only|preferably|applicants?))?\s*\)", re.I)
AGE = re.compile(r"^(\d{2})\s*[-\u2013]\s*(\d{2})(?:\s*(?:years?\s*old|yrs?\.?))?$", re.I)
TEMPLATE = PARAMS["template_default"]

def rule_rewrite(text, exp_spans, component):
    # returns (rewrite or None, rule id, status) with status in rewritten / review / no_rewrite / not_applicable
    if not exp_spans:
        return None, "none", "not_applicable"
    if is_listing(text):
        return text, "G1-listing", "no_rewrite"
    if COND_RX.search(text):
        return None, "gender-conditioned requirement", "review"
    if re.search(r"pregnan", text, re.I):
        return None, "pregnancy condition", "review"
    t, used = text, []
    for (s, e, c, w) in sorted(exp_spans, key=lambda x: -x[0]):
        if c == "E2":
            key = w.lower()
            if key not in E2_MAP:
                return None, "E2 ambiguous: " + key, "review"
            rep = E2_MAP[key]
            rep = rep.title() if component == "job_title" else (rep.capitalize() if w[:1].isupper() else rep)
            t = t[:s] + rep + t[e:]; used.append("T5")
        elif c == "E3":
            key = w.lower(); tail = t[e:e + 2]
            if key in ("his", "her") or tail in ("\u2019s", "'s"):
                t = t[:s] + "their" + (t[e + 2:] if tail in ("\u2019s", "'s") else t[e:])
            else:
                rest = t[e:]
                m = re.match(r"(\s+)(is|has|does|was)\b", rest)
                t = t[:s] + "they" + (m.group(1) + AGREE[m.group(2)] + rest[m.end():] if m else rest)
            used.append("T6")
    if any(c == "E1" for (_, _, c, _) in exp_spans):
        t2 = PAREN.sub("", t)
        rule = "T4" if t2 != t else "T1/T2"
        t2 = PHR.sub("", t2)
        t2 = re.sub(r"^[\s,;/&:\-\u2013]+|[\s,;/&:\-\u2013]+$", "", t2)
        t2 = re.sub(r"^(?:and|or)\s+", "", t2, flags=re.I)
        t2 = re.sub(r"\s{2,}", " ", t2).strip()
        if t2 == t.strip():
            return None, "E1 not removable", "review"
        if not t2:
            t, rule = TEMPLATE, "T1"
        else:
            m = AGE.match(t2)
            if m:
                t, rule = f"Candidates aged {m.group(1)}\u2013{m.group(2)} years old", "T3"
            else:
                t = t2[:1].upper() + t2[1:] if component != "job_title" else t2
                rule = "T4" if rule == "T4" else "T2"
        used.append(rule)
    return t, "+".join(dict.fromkeys(used)), "rewritten"

# template unit tests: the four examples of the brief (not corpus data) plus two more rule paths
demo = [("Male, 25\u201335 years old", "qualification", [(0, 4, "E1", "Male")]), ("Female, college graduate", "qualification", [(0, 6, "E1", "Female")]),
        ("Preferably Male", "qualification", [(0, 15, "E1", "Preferably Male")]), ("Salesman needed", "body_other", [(0, 8, "E2", "Salesman")]),
        ("Foreman Civil", "job_title", [(0, 7, "E2", "Foreman")]), ("Submit reports as he is responsible", "body_other", [(18, 20, "E3", "he")])]
print("Template unit tests (brief examples, not corpus data):")
for text, comp, spans in demo:
    print(f"  {text!r:42} -> {rule_rewrite(text, spans, comp)[0]!r}")

Template unit tests (brief examples, not corpus data):
  'Male, 25–35 years old'                    -> 'Candidates aged 25–35 years old'
  'Female, college graduate'                 -> 'College graduate'
  'Preferably Male'                          -> 'Open to all qualified candidates'
  'Salesman needed'                          -> 'Sales representative needed'
  'Foreman Civil'                            -> 'Foreperson Civil'
  'Submit reports as he is responsible'      -> 'Submit reports as they are responsible'


## 4. Semantic-preservation mechanism (the gate)

Every candidate rewrite, from any approach, must pass the same automatic gate before it is accepted. The brief asks for "a developed model" for semantic preservation, so the central check is the cosine similarity of sentence embeddings from `paraphrase-multilingual-MiniLM-L12-v2`, the encoder already used in the project (Feng *et al.* [11] support using a multilingual encoder when text can mix languages; this is background support for the choice of a multilingual model, not evidence about this model). Three rule-based checks sit next to it, because an embedding score cannot see a dropped number or a surviving gender word.

| Check | Rule | What it protects |
|---|---|---|
| G1 changed | the rewrite differs from the original | an unchanged unit has removed nothing |
| G2 no residual bias | no original explicit gender word remains; the S7 explicit rules find nothing; no coded stem (S8 word list) remains in a unit that had implicit wording, and none is added to an explicit-only unit | the bias is actually gone |
| G3 content retention | at least 0.80 of the original content words, excluding the flagged wording, appear in the rewrite | job requirements kept verbatim |
| G4 numbers | digits in the rewrite equal digits in the original | ages, years of experience, heights |
| G5 semantic similarity | cosine at least 0.70, except for a pure explicit gender restriction | meaning drift |
| G6 fluency proxy | not empty, no immediate repeated word, no chat preamble or line break, length ratio 0.4 to 2.0 when content remains | broken or chatty generations |

The **cosine and length checks are skipped only for a pure explicit gender restriction** ("Male", "Preferably Female"): the intended rewrite replaces the whole line, and any similarity between "Male" and "Open to all qualified candidates" is meaningless. They apply to every other unit, including an implicit unit whose words are all flagged. (A first full run skipped them whenever no content words remained, and the gate then accepted "Trustworthy; Enthusiastic" replaced by "Open to all applicants."; this was fixed after reading the test outputs and the whole notebook was re-run.)

The thresholds 0.70 and 0.80 are our design choices (no verified source). Section 10 shows how the results change with the cosine threshold. G2 re-runs the word lists that produced the silver labels, so passing G2 means "consistent with the rules", not "neutral to a reader". This follows the verify-and-flag design of Khondaker *et al.* [8] and the lexicon-based neutrality score of Kumar *et al.* [9].

In [5]:
from sentence_transformers import SentenceTransformer
S7LEX = json.load(open(S7 / "curated_explicit_lexicon.json", encoding="utf-8"))
S7_TERMS = set(S7LEX["terms"]); S7_PRON = set(S7LEX["pronouns_masculine"]) | set(S7LEX["pronouns_feminine"])
S7_SUFFIX = re.compile(S7LEX["occupational_suffix_pattern"]); S7_BARE = set(S7LEX["bare_occupational"]); S7_STOP = set(S7LEX["stop_list"])
TOKRX = re.compile(r"[A-Za-z0-9]+(?:[-'\u2019][A-Za-z0-9]+)*")
toks = lambda s: [w.lower() for w in TOKRX.findall(s)]

def s7_detect(text):
    # simplified S7 rules: R1 both-gender listing not flagged, R3 upper-case MAN not flagged, R4 pronoun pair not flagged, R5 suffix unless stop-listed
    words = TOKRX.findall(text); low = [w.lower() for w in words]
    male = any(w in S7_TERMS and S7LEX["terms"][w] == "M" for w in low); fem = any(w in S7_TERMS and S7LEX["terms"][w] == "F" for w in low)
    hits = []
    for w, l in zip(words, low):
        if l in S7_STOP:
            continue
        if l in S7_TERMS and not (male and fem):
            hits.append(l)
        elif l in S7_PRON and not (any(p in low for p in S7LEX["pronouns_masculine"]) and any(p in low for p in S7LEX["pronouns_feminine"])):
            hits.append(l)
        elif (l in S7_BARE or S7_SUFFIX.match(l)) and not (w.isupper() and len(w) > 1):
            hits.append(l)
    return hits

MASC = ("active", "adventurous", "aggress", "ambitio", "analy", "assert", "athlet", "autonom", "boast", "challeng", "compet", "confident", "courag",
        "decisive", "decision", "determin", "dominan", "driven", "fearless", "force", "headstrong", "hierarch", "impulsive", "independen", "individual",
        "intellect", "lead", "logic", "objective", "opinion", "outspoken", "persist", "self-relian", "self-sufficien", "stubborn", "superior", "unreasonab")
FEM = ("agree", "affectionate", "child", "cheer", "collab", "commit", "communal", "compassion", "connect", "considerate", "cooperat", "depend", "emotiona",
       "empath", "feel", "flatterable", "gentle", "honest", "interper", "interdependen", "kind", "kinship", "loyal", "modest", "nurtur", "pleasant",
       "polite", "quiet", "respons", "sensitiv", "submissive", "support", "sympath", "tender", "together", "trust", "understand", "warm", "enthusias", "yield", "share")
CODED_RX = re.compile(r"(?<!\w)(?:" + "|".join(re.escape(s) for s in sorted(MASC + FEM, key=len, reverse=True)) + r")[a-z-]*(?!\w)", re.I)
coded = lambda text: [m.group(0).lower() for m in CODED_RX.finditer(text) if not m.group(0).lower().startswith("confidential")]

STOP = set("a an and are as at be been by for from has have in into is it its of on or that the their them they this to was were will with who which can may must should shall also not no any all each per such than then there these those both either".split())
GENERIC = set("preferably only applicant applicants candidate candidates must be for".split())
embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", device="cpu")
_emb_cache = {}
def embed(texts):
    new = [t for t in dict.fromkeys(texts) if t not in _emb_cache]
    if new:
        for t, v in zip(new, embedder.encode(new, batch_size=64, normalize_embeddings=True, show_progress_bar=False)):
            _emb_cache[t] = v
    return [_emb_cache[t] for t in texts]

def flagged_tokens(tgt):
    return {w for (_, _, _, span) in tgt["exp"] + tgt["imp"] for w in toks(span)} | GENERIC

def gate(orig, rew, tgt, tau=None):
    tau = PARAMS["gate"]["cosine_min"] if tau is None else tau
    g = {"changed": rew is not None and rew.strip() != orig.strip()}
    if rew is None or not rew.strip():
        return {**g, "no_residual": False, "retention": np.nan, "numbers_ok": False, "cosine": np.nan, "fluent": False, "content_left": False, "residual_exp": True, "residual_imp": True, "pass": False}
    rt = set(toks(rew)); ot = toks(orig)
    exp_keys = {w for (_, _, c, span) in tgt["exp"] for w in toks(span) if w in S7_TERMS or w in S7_PRON or w in S7_BARE or S7_SUFFIX.match(w) or w in E2_MAP or w in ("man", "boy", "girl")}
    res_exp = bool(tgt["exp"]) and (any(k in rt for k in exp_keys) or bool(s7_detect(rew)))
    c_orig, c_rew = Counter(coded(orig)), Counter(coded(rew))
    res_imp = bool(c_rew) if tgt["imp"] else (bool(set(c_rew) - set(c_orig)))
    content = [w for w in ot if w not in STOP and w not in flagged_tokens(tgt) and not w.isdigit()]
    retention = (sum(w in rt for w in content) / len(content)) if content else 1.0
    nums_ok = sorted(re.findall(r"\d+", orig)) == sorted(re.findall(r"\d+", rew))
    content_left = not (bool(tgt["exp"]) and not tgt["imp"] and len(content) == 0)
    cos = float(np.dot(*embed([orig, rew]))) if content_left else np.nan
    words = rew.split()
    fluent = (not re.search(r"\n", rew.strip())) and not re.match(r"(?i)\s*(here is|here's|sure|rewritten|certainly)", rew) and not any(a.lower() == b.lower() for a, b in zip(words, words[1:]))
    if content_left:
        fluent = fluent and PARAMS["gate"]["length_ratio_range"][0] <= len(rew) / max(len(orig), 1) <= PARAMS["gate"]["length_ratio_range"][1]
    ok = g["changed"] and not res_exp and not res_imp and retention >= PARAMS["gate"]["content_retention_min"] and nums_ok and (not content_left or cos >= tau) and fluent
    return {**g, "no_residual": not (res_exp or res_imp), "residual_exp": res_exp, "residual_imp": res_imp, "retention": retention, "numbers_ok": nums_ok, "cosine": cos, "fluent": fluent, "content_left": content_left, "pass": bool(ok)}

# sanity: the original units must be flagged by the residual detectors, otherwise a pass in G2 would be meaningless
chk = tg[~tg.listing]
det_exp = chk[chk.n_exp > 0].apply(lambda r: bool(s7_detect(r.text)) or any(w.lower() in text_of[r.unit_id].lower() for (_, _, _, w) in TARGET[r.unit_id]["exp"]), axis=1)
det_imp = chk[chk.n_imp > 0].apply(lambda r: bool(coded(r.text)), axis=1)
print(f"original explicit units detected by the residual rules: {det_exp.mean():.3f} ({int(det_exp.sum())}/{len(det_exp)})")
print(f"original implicit units detected by the residual rules: {det_imp.mean():.3f} ({int(det_imp.sum())}/{len(det_imp)})")

D:\Dev\project-equal\equal-v2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Could not reach the Hub (Cannot reach https://huggingface.co/api/models/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2: offline mode is enabled. To disable it, please unset the `HF_HUB_OFFLINE` environment variable.). Using cached commit hash for 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3856.80it/s]

original explicit units detected by the residual rules: 1.000 (352/352)
original implicit units detected by the residual rules: 0.982 (584/595)


## 5. Approach 2 - fine-tuned sequence-to-sequence generation (`t5-small`)

The brief assigns this model to units where fixed templates give disfluent or incomplete rewrites, mainly multi-clause sentences with the biased token embedded. Amrhein *et al.* [3] train a seq2seq model to rewrite gender-biased text into gender-fair text, and Mainardi *et al.* [4] report that small fine-tuned models do well with limited data; both study German or Italian text and neither uses job ads.

**What the training pairs are.** The corpus contains no human-written neutral rewrites, so the training pairs are **silver**: each pair is a training-pool explicit unit and the Approach 1 rewrite of it, kept only if that rewrite passes the gate. We add 250 unchanged units with no flagged wording as identity pairs, so the model learns to copy when nothing is wrong. **Implicit units have no target, so they are not in the training data.** This makes Approach 2 a student of the rules: it can only learn what the rules already do, and an agreement with the rules is not evidence of rewrite quality. Pairs are de-duplicated by source text. The share of test explicit texts that also occur in the training pairs is reported, because for those texts Approach 2 is repeating training data.

Model `google-t5/t5-small`, prefix `neutralize: `, AdamW, learning rate 3e-4, weight decay 0.01, batch 16, 20 epochs, three seeds (1, 2, 3), beam search with 4 beams. No epoch or checkpoint selection is done.

In [6]:
def a1_for(u):
    t = TARGET[u]
    return rule_rewrite(text_of[u], t["exp"], comp_of[u])

# A1 rewrites for every target unit
A1 = {u: a1_for(u) for u in tg["unit_id"]}
tg["a1_status"] = tg["unit_id"].map(lambda u: A1[u][2]); tg["a1_rule"] = tg["unit_id"].map(lambda u: A1[u][1])
print(tg[tg.split == "test"].groupby(["kind", "a1_status"]).size().unstack(fill_value=0))

# silver training pairs: training-pool explicit units whose rule rewrite passes the gate
pairs = {}
for r in tg[(tg.split == "train") & (tg.n_exp > 0)].itertuples():
    rew, rule, status = A1[r.unit_id]
    if status == "no_rewrite":
        pairs.setdefault(r.text, (r.text, "identity (listing)"))
    elif status == "rewritten" and not r.n_imp and gate(r.text, rew, TARGET[r.unit_id])["pass"]:
        pairs.setdefault(r.text, (rew, rule))
flagged_units = set(tg["unit_id"])
free = units[(units["record_id"].isin(train_ads)) & (~units["unit_id"].isin(flagged_units)) & (units["text_masked"].astype(str).str.len() > 3)]
ident = free.sample(PARAMS["t5"]["identity_pairs"], random_state=PARAMS["t5"]["identity_seed"])["text_masked"].astype(str).tolist()
for t in ident:
    pairs.setdefault(t, (t, "identity (no flagged wording)"))
pair_df = pd.DataFrame([{"source": s, "target": v[0], "rule": v[1]} for s, v in pairs.items()])
pair_df.to_csv(OUT / "t5_training_pairs.csv", index=False)
print(f"training pairs: {len(pair_df)} distinct sources; changed pairs {int((pair_df.source != pair_df.target).sum())}; identity pairs {int((pair_df.source == pair_df.target).sum())}")
print(pair_df[pair_df.source != pair_df.target].rule.value_counts().to_dict())
test_exp_texts = set(tg[(tg.split == "test") & (tg.n_exp > 0) & ~tg.listing].text)
print(f"distinct test explicit texts: {len(test_exp_texts)}; also present as a training source: {len(test_exp_texts & set(pair_df.source))}")

a1_status  not_applicable  review  rewritten
kind                                        
both                    0       0          2
explicit                0       3         91
implicit              153       0          0


training pairs: 267 distinct sources; changed pairs 28; identity pairs 239
{'T5': 17, 'T1': 9, 'T4': 1, 'T2': 1}
distinct test explicit texts: 19; also present as a training source: 9


In [7]:
import torch
from transformers import AutoTokenizer, T5ForConditionalGeneration
torch.set_num_threads(max(1, (os.cpu_count() or 4) - 1))
T5C = PARAMS["t5"]
tok5 = AutoTokenizer.from_pretrained(T5C["model"])
eval_units = tg[(tg.split == "test") & (~tg.listing)].reset_index(drop=True)
eval_texts = sorted(set(eval_units["text"]))
T5_FILE = OUT / "t5_predictions.json"

def train_t5(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    model = T5ForConditionalGeneration.from_pretrained(T5C["model"]); model.train()
    opt = torch.optim.AdamW(model.parameters(), lr=T5C["lr"], weight_decay=T5C["weight_decay"])
    data = list(zip(pair_df["source"], pair_df["target"])); log = []
    for ep in range(1, T5C["epochs"] + 1):
        random.shuffle(data); tot = 0.0; t0 = time.time()
        for i in range(0, len(data), T5C["batch"]):
            b = data[i:i + T5C["batch"]]
            x = tok5([T5C["prefix"] + s for s, _ in b], return_tensors="pt", padding=True, truncation=True, max_length=T5C["max_source_len"])
            y = tok5([t for _, t in b], return_tensors="pt", padding=True, truncation=True, max_length=T5C["max_target_len"]).input_ids
            y[y == tok5.pad_token_id] = -100
            loss = model(**x, labels=y).loss; loss.backward(); opt.step(); opt.zero_grad(); tot += float(loss) * len(b)
        log.append({"seed": seed, "epoch": ep, "train_loss": round(tot / len(data), 4), "seconds": round(time.time() - t0, 1)})
    model.eval(); preds = {}
    with torch.no_grad():
        for i in range(0, len(eval_texts), 32):
            b = eval_texts[i:i + 32]
            x = tok5([T5C["prefix"] + s for s in b], return_tensors="pt", padding=True, truncation=True, max_length=T5C["max_source_len"])
            out = model.generate(**x, num_beams=T5C["num_beams"], max_new_tokens=T5C["max_target_len"])
            for s, o in zip(b, tok5.batch_decode(out, skip_special_tokens=True)):
                preds[s] = o
    return preds, log

if T5_FILE.exists():
    T5_OUT = json.load(open(T5_FILE, encoding="utf-8"))
else:
    T5_OUT = {"preds": {}, "log": []}
    for sd in T5C["seeds"]:
        p, lg = train_t5(sd); T5_OUT["preds"][str(sd)] = p; T5_OUT["log"] += lg
        json.dump(T5_OUT, open(T5_FILE, "w", encoding="utf-8"))
pd.DataFrame(T5_OUT["log"]).to_csv(OUT / "t5_training_log.csv", index=False)
lg = pd.DataFrame(T5_OUT["log"]); print(lg[lg.epoch.isin([1, 5, 10, 20, T5C["epochs"]])].drop_duplicates(["seed", "epoch"]).to_string(index=False))

 seed  epoch  train_loss  seconds
    1      1      0.4256     26.1
    1      5      0.0472     24.0
    1     10      0.0162     25.9
    1     20      0.0101     25.2
    2      1      0.4995     25.6
    2      5      0.0517     24.4
    2     10      0.0208     25.1
    2     20      0.0062     23.8
    3      1      0.4252     25.6
    3      5      0.0389     27.3
    3     10      0.0213     24.7
    3     20      0.0117     25.5


## 6. Approach 3 - LLM-guided rewriting (`Qwen2.5-1.5B-Instruct`, not GPT-4)

For every target unit the model receives the original line, the list of flagged words and the three constraints from the brief: (1) preserve all factual job requirements verbatim, (2) eliminate the identified gender-preferential language, (3) maintain the original sentence structure. It also receives the project's own template for a line that is only a gender requirement. No worked examples are given, so no test text leaks into the prompt.

Decoding is greedy, so the same prompt should give the same output. Blackwell *et al.* [10] report that LLM output can vary even at fixed settings (preprint), so the first six distinct prompts are generated a second time and compared. Prompting without retraining for debiasing [6] and for minimal edits [7] supports this approach in general, and Shu *et al.* [5] study instruction-controlled rewriting; none of them is about job-ad bias or a 1.5-billion-parameter model, and none states the result a small model should reach.

In [8]:
from transformers import AutoModelForCausalLM
LC = PARAMS["llm"]
LLM_FILE = OUT / "llm_outputs.json"
SYSTEM = ("You rewrite one line of a job advertisement so that it contains no gender-preferential language. "
          "Rules: (1) preserve all factual job requirements verbatim, including numbers, qualifications and skills; "
          "(2) eliminate the identified gender-preferential language; (3) maintain the original sentence structure. "
          "If the whole line is only a gender requirement, answer exactly: " + TEMPLATE + ". "
          "Answer with the rewritten line only, with no explanation.")
def prompt_for(u):
    t = TARGET[u]; words = list(dict.fromkeys([w for (_, _, _, w) in t["exp"] + t["imp"]]))
    return f"Line: {text_of[u]}\nIdentified wording to remove or neutralise: {', '.join(words)}"
llm_key = lambda u: prompt_for(u)
need = sorted(set(llm_key(u) for u in eval_units["unit_id"]))
LLM = json.load(open(LLM_FILE, encoding="utf-8")) if LLM_FILE.exists() else {"out": {}, "rerun": {}}
todo = [k for k in need if k not in LLM["out"]]
if SMOKE:
    todo = todo[:12]
if todo:
    QWEN_DIR = str(next((Path.home() / ".cache" / "huggingface" / "hub" / "models--Qwen--Qwen2.5-1.5B-Instruct" / "snapshots").iterdir()))
    tokq = AutoTokenizer.from_pretrained(QWEN_DIR); mq = AutoModelForCausalLM.from_pretrained(QWEN_DIR, dtype=torch.float32); mq.eval()
    def gen(user):
        msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]
        ids = tokq.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True)
        with torch.no_grad():
            o = mq.generate(**ids, max_new_tokens=LC["max_new_tokens"], do_sample=False, pad_token_id=tokq.eos_token_id)
        return tokq.decode(o[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    t0 = time.time()
    for i, k in enumerate(todo):
        LLM["out"][k] = gen(k)
        if (i + 1) % 10 == 0:
            json.dump(LLM, open(LLM_FILE, "w", encoding="utf-8")); print(f"  {i+1}/{len(todo)} prompts, {time.time()-t0:.0f}s")
    for k in need[:6]:
        LLM["rerun"][k] = gen(k)
    json.dump(LLM, open(LLM_FILE, "w", encoding="utf-8"))
print(f"distinct prompts: {len(need)}; cached outputs: {len(LLM['out'])}")
same = [LLM["out"][k] == LLM["rerun"].get(k) for k in need[:6] if k in LLM["rerun"]]
print(f"determinism re-run: {sum(same)} of {len(same)} identical")

distinct prompts: 154; cached outputs: 154
determinism re-run: 6 of 6 identical


## 7. Cascade

For each target unit the cascade tries the approaches in the brief's order and accepts the **first candidate that passes the gate**: Approach 1 (explicit units the rules can handle), then Approach 2 (seed 1), then Approach 3. If no candidate passes, the unit is left unchanged and flagged for review. This is a fixed generate-then-verify pipeline without retries, in the spirit of [8]. Approach 2 and 3 are also scored **alone** on every target unit, so their individual quality does not depend on what the earlier stage left over.

In [9]:
SYSTEMS = ["A1 rules", "A2 t5-small (3 seeds)", "A3 Qwen2.5-1.5B"]
rec = []
seeds = [str(s) for s in PARAMS["t5"]["seeds"]]
for r in eval_units.itertuples():
    u, tgt, orig = r.unit_id, TARGET[r.unit_id], r.text
    cand = {}
    rew1, rule1, st1 = A1[u]
    cand["A1 rules"] = [rew1] if st1 == "rewritten" else [None]
    cand["A2 t5-small (3 seeds)"] = [T5_OUT["preds"][s].get(orig) for s in seeds]
    cand["A3 Qwen2.5-1.5B"] = [LLM["out"].get(llm_key(u))]
    for sysn, lst in cand.items():
        for j, rew in enumerate(lst):
            g = gate(orig, rew, tgt)
            rec.append({"unit_id": u, "record_id": r.record_id, "kind": r.kind, "role_noun_only": r.role_noun_only, "system": sysn, "variant": j, "rewrite": rew,
                        "available": rew is not None, **{k: v for k, v in g.items()}})
cand_df = pd.DataFrame(rec)
cand_df["nearest_seed_cos"] = cand_df["cosine"]
print(cand_df.groupby("system").agg(candidates=("unit_id", "size"), available=("available", "sum"), changed=("changed", "sum"), passed=("pass", "sum")))

# cascade (A2 seed 1 is variant 0)
dec = []
for r in eval_units.itertuples():
    u = r.unit_id; chosen, why = None, "review"
    if A1[u][2] == "review" and A1[u][1] in ("gender-conditioned requirement", "pregnancy condition"):
        dec.append({"unit_id": u, "accepted_by": "review", "final_text": r.text, "review_reason": "manual review: " + A1[u][1]}); continue
    for sysn, var in (("A1 rules", 0), ("A2 t5-small (3 seeds)", 0), ("A3 Qwen2.5-1.5B", 0)):
        row = cand_df[(cand_df.unit_id == u) & (cand_df.system == sysn) & (cand_df.variant == var)].iloc[0]
        if row["pass"]:
            chosen, why = row["rewrite"], sysn; break
    dec.append({"unit_id": u, "accepted_by": why, "final_text": chosen if chosen is not None else r.text, "review_reason": "" if chosen is not None else "no candidate passed the gate"})
dec = pd.DataFrame(dec)
eval_units = eval_units.merge(dec, on="unit_id")
print(eval_units.groupby(["kind", "accepted_by"]).size().unstack(fill_value=0))

                       candidates  available  changed  passed
system                                                       
A1 rules                      249         93       93      91
A2 t5-small (3 seeds)         747        747      270     267
A3 Qwen2.5-1.5B               249        249      238     147


accepted_by  A1 rules  A3 Qwen2.5-1.5B  review
kind                                          
both                0                0       2
explicit           91                1       2
implicit            0               57      96


## 8. Evaluation design

**Unit of analysis.** The target units of the 400 test advertisements, excluding the `no_rewrite` listings. **Reference.** There is none for rewrite quality; all measures are properties of the rewrite checked by the gate. **Measures per system and unit kind** (explicit, implicit, both):
- *candidate rate*: the system produced a changed candidate;
- *bias removal*: no flagged wording remains (an unchanged unit counts as not removed);
- *gate pass*: removal plus content retention, numbers, similarity and fluency proxy together;
- *preservation among changed*: median cosine and median content retention of changed candidates.

Intervals are 95% bootstrap percentile intervals over advertisements (1,000 resamples, seed 7), the same design as Steps 7 and 8. For Approach 2 the three seeds are averaged before resampling. A "readability" proxy (Flesch reading ease with a vowel-group syllable heuristic) is reported at advertisement level in Section 9 only; on short fragments it is unreliable.

In [10]:
rngb = np.random.RandomState(PARAMS["bootstrap_seed"]); B = PARAMS["bootstrap_resamples"]
def per_unit(system):
    d = cand_df[cand_df.system == system]
    agg = d.groupby("unit_id").agg(avail=("available", "mean"), changed=("changed", "mean"), removal=("no_residual", "mean"), passed=("pass", "mean"),
                                   cos=("cosine", "median"), ret=("retention", "median"))
    return agg
tbl_units = eval_units.set_index("unit_id")
PU = {s: per_unit(s).reindex(tbl_units.index) for s in SYSTEMS}
casc = pd.DataFrame({"avail": 1.0, "changed": (tbl_units.accepted_by != "review").astype(float), "removal": (tbl_units.accepted_by != "review").astype(float),
                     "passed": (tbl_units.accepted_by != "review").astype(float)}, index=tbl_units.index)
PU["Cascade A1>A2>A3"] = casc

def boot_ratio(mask, num, den=None):
    ads = tbl_units.loc[mask, "record_id"].values; n = pd.Series(num[mask].values, index=ads); d = pd.Series(1.0 if den is None else den[mask].values, index=ads) if den is not None else pd.Series(1.0, index=ads)
    ns, ds = n.groupby(level=0).sum(), d.groupby(level=0).sum(); idx = ns.index.values
    pt = ns.sum() / ds.sum() if ds.sum() else np.nan
    bs = []
    for _ in range(B):
        pick = rngb.randint(0, len(idx), len(idx)); den_ = ds.values[pick].sum()
        bs.append(ns.values[pick].sum() / den_ if den_ else np.nan)
    return pt, np.nanpercentile(bs, 2.5), np.nanpercentile(bs, 97.5)

rows = []
groups = {"explicit": tbl_units.kind == "explicit", "implicit": tbl_units.kind == "implicit", "both": tbl_units.kind == "both", "all": tbl_units.kind.notna()}
for sysn, pu in PU.items():
    for gname, mask in groups.items():
        if mask.sum() == 0:
            continue
        for metric, col in (("candidate rate (changed)", "changed"), ("bias removal", "removal"), ("gate pass", "passed")):
            v = pu[col].fillna(0.0); pt, lo, hi = boot_ratio(mask, v)
            rows.append({"system": sysn, "group": gname, "metric": metric, "value": pt, "ci_low": lo, "ci_high": hi, "n_units": int(mask.sum())})
        if "cos" in pu:
            m2 = mask & (pu["changed"].fillna(0) > 0)
            rows.append({"system": sysn, "group": gname, "metric": "median cosine (changed, content left)", "value": float(pu.loc[m2, "cos"].median()), "ci_low": np.nan, "ci_high": np.nan, "n_units": int(m2.sum())})
            rows.append({"system": sysn, "group": gname, "metric": "median content retention (changed)", "value": float(pu.loc[m2, "ret"].median()), "ci_low": np.nan, "ci_high": np.nan, "n_units": int(m2.sum())})
res = pd.DataFrame(rows); res.to_csv(OUT / "results_long.csv", index=False)
fmt = lambda r: f"{r.value:.3f}" + (f" ({r.ci_low:.3f}-{r.ci_high:.3f})" if pd.notna(r.ci_low) else "")
res["cell"] = res.apply(fmt, axis=1)
table = res.pivot_table(index=["group", "metric"], columns="system", values="cell", aggfunc="first")
table = table[[c for c in ["A1 rules", "A2 t5-small (3 seeds)", "A3 Qwen2.5-1.5B", "Cascade A1>A2>A3"] if c in table.columns]]
table.to_csv(OUT / "results_table.csv"); print(table.to_string())
print("\nunits per group:", {g: int(m.sum()) for g, m in groups.items()})

system                                                     A1 rules A2 t5-small (3 seeds)      A3 Qwen2.5-1.5B     Cascade A1>A2>A3
group    metric                                                                                                                    
all      bias removal                           0.365 (0.307-0.427)   0.369 (0.312-0.428)  0.763 (0.711-0.820)  0.598 (0.537-0.658)
         candidate rate (changed)               0.373 (0.312-0.440)   0.361 (0.301-0.426)  0.956 (0.927-0.982)  0.598 (0.538-0.661)
         gate pass                              0.365 (0.299-0.430)   0.357 (0.295-0.423)  0.590 (0.525-0.654)  0.598 (0.540-0.664)
         median content retention (changed)                   1.000                 1.000                1.000                  NaN
         median cosine (changed, content left)                0.882                 0.845                0.900                  NaN
both     bias removal                           0.000 (0.000-0.000)   0.000 

### 8.1 Paired differences and Approach 2 on seen and unseen text

The gate pass rates of the approaches are compared on the **same units** (paired), with the same advertisement bootstrap. For Approach 2, explicit test texts that also occur among the training sources are separated from texts it has not seen.

In [11]:
pairs_cmp = []
for a, b in (("A3 Qwen2.5-1.5B", "A2 t5-small (3 seeds)"), ("A2 t5-small (3 seeds)", "A1 rules"), ("Cascade A1>A2>A3", "A3 Qwen2.5-1.5B")):
    for gname in ("explicit", "implicit", "all"):
        mask = groups[gname]
        if mask.sum() == 0:
            continue
        diff = PU[a]["passed"].fillna(0) - PU[b]["passed"].fillna(0)
        if gname == "implicit" and "A1" in b:
            continue
        pt, lo, hi = boot_ratio(mask, diff)
        pairs_cmp.append({"comparison": f"{a} minus {b}", "group": gname, "metric": "gate pass", "difference": pt, "ci_low": lo, "ci_high": hi, "excludes_0": not (lo <= 0 <= hi)})
pc = pd.DataFrame(pairs_cmp); pc.to_csv(OUT / "paired_differences.csv", index=False); print(pc.round(3).to_string(index=False))

train_src = set(pair_df.source)
tbl_units["seen_text"] = tbl_units["text"].isin(train_src)
ex = tbl_units[tbl_units.kind == "explicit"]
seen_tbl = pd.DataFrame({"units": ex.groupby("seen_text").size(),
                         "A1 gate pass": PU["A1 rules"].loc[ex.index, "passed"].fillna(0).groupby(ex.seen_text).mean(),
                         "A2 gate pass": PU["A2 t5-small (3 seeds)"].loc[ex.index, "passed"].fillna(0).groupby(ex.seen_text).mean(),
                         "A3 gate pass": PU["A3 Qwen2.5-1.5B"].loc[ex.index, "passed"].fillna(0).groupby(ex.seen_text).mean()}).round(3)
print("\nexplicit test units by whether their text is a training source of Approach 2:"); print(seen_tbl)
seen_tbl.to_csv(OUT / "a2_seen_vs_unseen.csv")

                                 comparison    group    metric  difference  ci_low  ci_high  excludes_0
A3 Qwen2.5-1.5B minus A2 t5-small (3 seeds) explicit gate pass       0.011  -0.033    0.062       False
A3 Qwen2.5-1.5B minus A2 t5-small (3 seeds) implicit gate pass       0.373   0.290    0.455        True
A3 Qwen2.5-1.5B minus A2 t5-small (3 seeds)      all gate pass       0.233   0.172    0.296        True
       A2 t5-small (3 seeds) minus A1 rules explicit gate pass      -0.021  -0.053    0.000       False
       A2 t5-small (3 seeds) minus A1 rules      all gate pass      -0.008  -0.020    0.000       False
     Cascade A1>A2>A3 minus A3 Qwen2.5-1.5B explicit gate pass       0.021  -0.022    0.063       False
     Cascade A1>A2>A3 minus A3 Qwen2.5-1.5B implicit gate pass       0.000   0.000    0.000       False
     Cascade A1>A2>A3 minus A3 Qwen2.5-1.5B      all gate pass       0.008  -0.008    0.024       False



explicit test units by whether their text is a training source of Approach 2:
           units  A1 gate pass  A2 gate pass  A3 gate pass
seen_text                                                 
False          8         0.625         0.375         0.625
True          86         1.000         1.000         0.988


## 9. Advertisement-level effect of the cascade

The cascade's accepted rewrites replace their units in the 400 test advertisements; every other unit, including the `no_rewrite` listings and the units sent to review, is left as it was. An advertisement is **fully neutralised** when every target unit in it was accepted. The side-by-side original and rewritten advertisements are saved in `rewritten_ads_test.csv`, in the layout of the evaluation table in the brief (the human rating columns are intentionally absent).

In [12]:
final_of = dict(zip(eval_units.unit_id, eval_units.final_text)); acc_of = dict(zip(eval_units.unit_id, eval_units.accepted_by))
nr_units = set(tg[(tg.split == "test") & tg.listing].unit_id)
def fre(texts):
    words = sum(len(t.split()) for t in texts); sents = max(len(texts), 1)
    syl = sum(max(1, len(re.findall(r"[aeiouy]+", w.lower()))) for t in texts for w in t.split())
    return 206.835 - 1.015 * words / sents - 84.6 * syl / max(words, 1) if words else np.nan
ad_rows, sbs = [], []
for ad in test_ads:
    us = units[units.record_id == ad].sort_values("unit_index")
    orig = [str(t) for t in us.text_masked]; new = [final_of.get(u, t) for u, t in zip(us.unit_id, orig)]
    tgt_ids = [u for u in us.unit_id if u in acc_of]
    if not tgt_ids:
        continue
    acc = [acc_of[u] != "review" for u in tgt_ids]
    kinds = {tbl_units.loc[u, "kind"] for u in tgt_ids}
    ad_rows.append({"record_id": ad, "route_v03": route_of[ad], "target_units": len(tgt_ids), "accepted": int(sum(acc)), "review": int(len(acc) - sum(acc)),
                    "fully_neutralised": all(acc), "kinds": "+".join(sorted(kinds)), "fre_before": fre(orig), "fre_after": fre(new),
                    "chars_before": sum(map(len, orig)), "chars_after": sum(map(len, new))})
    for u, o, n in zip(us.unit_id, orig, new):
        if u in acc_of:
            sbs.append({"record_id": ad, "unit_id": u, "component": comp_of[u], "original": o, "rewritten": n, "accepted_by": acc_of[u], "status": "accepted" if acc_of[u] != "review" else "needs review (unchanged)"})
adm = pd.DataFrame(ad_rows); sbs = pd.DataFrame(sbs)
adm.to_csv(OUT / "ad_level_results.csv", index=False); sbs.to_csv(OUT / "rewritten_ads_test.csv", index=False)
print(f"advertisements with at least one rewrite target (excluding listing-only ads): {len(adm)}")
summ = adm.groupby("kinds").agg(ads=("record_id", "size"), fully_neutralised=("fully_neutralised", "mean"), median_target_units=("target_units", "median")).round(3)
print(summ)
print(f"overall fully neutralised: {adm.fully_neutralised.mean():.3f} ({int(adm.fully_neutralised.sum())}/{len(adm)}); units accepted {int(adm.accepted.sum())} of {int(adm.target_units.sum())}")
ptn, lo, hi = boot_ratio(tbl_units.kind.notna(), (tbl_units.accepted_by != "review").astype(float))
print(f"unit-level cascade acceptance: {ptn:.3f} (95% CI {lo:.3f}-{hi:.3f})")
print(f"readability proxy (Flesch, heuristic) on ads with a rewrite: median before {adm.fre_before.median():.1f}, after {adm.fre_after.median():.1f}; median change {np.nanmedian(adm.fre_after-adm.fre_before):+.1f}")

advertisements with at least one rewrite target (excluding listing-only ads): 183
                   ads  fully_neutralised  median_target_units
kinds                                                         
both                 1              0.000                  1.0
both+explicit        1              0.000                  3.0
explicit            74              0.986                  1.0
explicit+implicit   15              0.200                  2.0
implicit            92              0.272                  1.0
overall fully neutralised: 0.552 (101/183); units accepted 149 of 249
unit-level cascade acceptance: 0.598 (95% CI 0.532-0.664)
readability proxy (Flesch, heuristic) on ads with a rewrite: median before 24.1, after 23.6; median change +0.0


In [13]:
# example accepted and rejected rewrites, taken directly from the outputs
show = eval_units.merge(cand_df[(cand_df.system == "A3 Qwen2.5-1.5B")][["unit_id", "rewrite", "pass", "cosine", "retention", "residual_exp", "residual_imp"]], on="unit_id", how="left")
show = show.rename(columns={"rewrite": "A3 rewrite", "pass": "A3 gate pass"})
for k in ("explicit", "implicit"):
    s = show[show.kind == k].drop_duplicates("text")
    print(f"\n=== {k}: accepted by cascade (first 8 distinct texts) ===")
    for r in s[s.accepted_by != "review"].head(8).itertuples():
        print(f"  [{r.accepted_by}] {r.text[:90]!r}\n      -> {r.final_text[:90]!r}")
    print(f"=== {k}: left for review (first 8 distinct texts), with the A3 output and why it failed ===")
    for _, r in s[s.accepted_by == "review"].head(8).iterrows():
        why = [n for n, f in (("residual explicit", r["residual_exp"]), ("residual coded wording", r["residual_imp"])) if f is True]
        cs = "" if pd.isna(r["cosine"]) else f"cos {r['cosine']:.2f}"
        print(f"  {r['text'][:90]!r}\n      A3: {str(r['A3 rewrite'])[:90]!r} | {'; '.join(why) or 'other gate'} {cs}")


=== explicit: accepted by cascade (first 8 distinct texts) ===
  [A1 rules] 'Male'
      -> 'Open to all qualified candidates'
  [A1 rules] 'Female'
      -> 'Open to all qualified candidates'
  [A1 rules] 'Must weigh not more or less than five kilograms (5kgs) from the standard weight correspond'
      -> 'Must weigh not more or less than five kilograms (5kgs) from the standard weight correspond'
  [A1 rules] 'Foreman Mechanical'
      -> 'Foreperson Mechanical'
  [A1 rules] "With at least three (3) years' work experience as Foreman"
      -> "With at least three (3) years' work experience as Foreperson"
  [A1 rules] 'Foreman Plumbing'
      -> 'Foreperson Plumbing'
  [A1 rules] 'Preferably male'
      -> 'Open to all qualified candidates'
  [A1 rules] 'Saleslady'
      -> 'Sales Representative'
=== explicit: left for review (first 8 distinct texts), with the A3 output and why it failed ===
  'Female SRs must not be pregnant'
      A3: 'All qualified candidates must not be pregnant' 

## 10. Error analysis and sensitivity checks

In [14]:
# (a) why do candidates fail the gate? first failed check per candidate, by system and kind
def first_fail(r):
    if not r["available"]: return "no candidate"
    if not r["changed"]: return "unchanged"
    if r["residual_exp"] or r["residual_imp"]: return "bias wording remains"
    if r["retention"] < PARAMS["gate"]["content_retention_min"]: return "content dropped"
    if not r["numbers_ok"]: return "numbers changed"
    if r["content_left"] and not (r["cosine"] >= PARAMS["gate"]["cosine_min"]): return "similarity below threshold"
    if not r["fluent"]: return "fluency proxy"
    return "pass"
cand_df["first_fail"] = cand_df.apply(first_fail, axis=1)
ff = cand_df.groupby(["system", "kind", "first_fail"]).size().unstack(fill_value=0)
print(ff.to_string()); ff.to_csv(OUT / "first_failed_check.csv")

# (b) cosine-threshold sensitivity for the gate pass rate (explicit and implicit separately)
sens = []
for tau in PARAMS["tau_grid"]:
    for sysn in ("A2 t5-small (3 seeds)", "A3 Qwen2.5-1.5B"):
        d = cand_df[cand_df.system == sysn]
        ok = []
        for r in d.itertuples():
            tgt = TARGET[r.unit_id]
            if not r.available:
                ok.append(False); continue
            ok.append(bool(r.changed and r.no_residual and r.retention >= PARAMS["gate"]["content_retention_min"] and r.numbers_ok and (not r.content_left or r.cosine >= tau) and r.fluent))
        d = d.assign(ok=ok)
        for k in ("explicit", "implicit", "both"):
            dd = d[d.kind == k]
            if len(dd): sens.append({"tau": tau, "system": sysn, "kind": k, "gate_pass": dd.ok.mean()})
sens = pd.DataFrame(sens); sens.to_csv(OUT / "tau_sensitivity.csv", index=False)
print("\ngate pass rate by cosine threshold:"); print(sens.pivot_table(index=["system", "kind"], columns="tau", values="gate_pass").round(3).to_string())

# (c) role-noun-only implicit units dropped
m = tbl_units.kind == "implicit"; m2 = m & ~tbl_units.role_noun_only
print(f"\nimplicit units: {int(m.sum())}; excluding role-noun-only silver units: {int(m2.sum())}")
for s in SYSTEMS + ["Cascade A1>A2>A3"]:
    print(f"  {s:26} gate pass all implicit {PU[s].loc[m, 'passed'].fillna(0).mean():.3f} | excluding role nouns {PU[s].loc[m2, 'passed'].fillna(0).mean():.3f}")

# (d) by component
print("\ncascade acceptance by component:"); print(eval_units.assign(acc=eval_units.accepted_by != "review").groupby(["component", "kind"]).agg(units=("acc", "size"), accepted=("acc", "mean")).round(3).to_string())

# (e) does an accepted rewrite only delete words? A pure deletion removes the flagged word but may also remove a requirement the gate cannot see.
acc_all = eval_units[eval_units.accepted_by != "review"].copy()
acc_all["deletion_only"] = [set(toks(f)) <= set(toks(o)) and len(f) < len(o) for o, f in zip(acc_all.text, acc_all.final_text)]
print("\naccepted rewrites that only delete words (no word added):")
print(acc_all.groupby(["kind", "accepted_by"]).agg(accepted=("deletion_only", "size"), deletion_only=("deletion_only", "sum")).to_string())
imp_acc = acc_all[acc_all.kind == "implicit"]
print("\naccepted implicit rewrites (distinct):")
for r in imp_acc.drop_duplicates("text").itertuples():
    print(f"  {'[deletion] ' if r.deletion_only else '[replaced] '}{r.text[:80]!r} -> {r.final_text[:80]!r}")
acc_all[["unit_id", "kind", "accepted_by", "deletion_only"]].to_csv(OUT / "accepted_deletion_only.csv", index=False)

first_fail                      bias wording remains  content dropped  no candidate  pass  similarity below threshold  unchanged
system                kind                                                                                                      
A1 rules              both                         2                0             0     0                           0          0
                      explicit                     0                0             3    91                           0          0
                      implicit                     0                0           153     0                           0          0
A2 t5-small (3 seeds) both                         0                0             0     0                           0          6
                      explicit                     0                0             0   267                           0         15
                      implicit                     3                0             0     0        

## 11. Validation checks

In [15]:
checks = []
add = lambda name, ok, detail="": checks.append({"check": name, "passed": bool(ok), "detail": detail})
add("test sample equals the Step 7 file and has 400 ads", len(set(test_ads)) == 400 and set(test_ads) == set(rd(S7 / "test_sample_400_ads.csv").record_id))
add("no test ad in training or validation", not (set(train_ads) | val_ads) & set(test_ads))
add("every target span reproduces inside its unit text", len(bad) == 0, f"{len(bad)} mismatches")
add("residual rules flag at least 95% of the original explicit units", det_exp.mean() >= 0.95, f"{det_exp.mean():.3f}")
add("residual rules flag at least 95% of the original implicit units", det_imp.mean() >= 0.95, f"{det_imp.mean():.3f}")
add("no test unit id is a training source of Approach 2", not (set(tg[tg.split == 'test'].unit_id) & set(tg[(tg.split == 'train')].unit_id)))
add("T5 training sources come from training-pool ads only", all(s in set(units[units.record_id.isin(train_ads)].text_masked.astype(str)) for s in pair_df.source))
add("all target units have a cascade decision", len(dec) == len(eval_units))
acc_rows = eval_units[eval_units.accepted_by != "review"]
add("every accepted rewrite passes the gate when re-checked", all(gate(r.text, r.final_text, TARGET[r.unit_id])["pass"] for r in acc_rows.itertuples()))
add("no accepted rewrite changes any number", all(sorted(re.findall(r"\d+", r.text)) == sorted(re.findall(r"\d+", r.final_text)) for r in acc_rows.itertuples()))
add("no accepted rewrite keeps an original explicit gender word", all(not (set(toks(r.final_text)) & {w for (_, _, _, sp_) in TARGET[r.unit_id]['exp'] for w in toks(sp_) if w in S7_TERMS or w in S7_PRON or w in E2_MAP or w in ('man', 'boy')}) for r in acc_rows.itertuples()))
rev_rows = eval_units[eval_units.accepted_by == "review"]
add("units sent to review are left unchanged", all(r.text == r.final_text for r in rev_rows.itertuples()))
add("no_rewrite listings are unchanged in the side-by-side file", all(sbs[sbs.unit_id.isin(nr_units)].original == sbs[sbs.unit_id.isin(nr_units)].rewritten) if len(sbs) else True)
add("side-by-side file contains only target units whose unchanged status matches the decision", all((r.original == r.rewritten) == (r.accepted_by == "review") or r.original == r.rewritten for r in sbs.itertuples()))
add("LLM outputs cached for every prompt", all(k in LLM["out"] for k in need), f"{len(need)} prompts")
add("determinism re-run: all re-generated outputs identical", all(same) if same else True, f"{sum(same)}/{len(same)}")
add("ad-level table covers every ad with a target unit", len(adm) == tg[(tg.split == 'test') & (~tg.listing)].record_id.nunique())
add("results table has every system and group", set(res.system) == set(PU) and set(res.group) <= set(groups))
add("all cosine values lie in [-1, 1]", cand_df.cosine.dropna().between(-1.0001, 1.0001).all())
vc = pd.DataFrame(checks); vc.to_csv(OUT / "validation_checks.csv", index=False)
print(vc.to_string(index=False)); print(f"\n{int(vc.passed.sum())} of {len(vc)} checks passed")

cand_df.drop(columns=["nearest_seed_cos"]).to_csv(OUT / "candidate_rewrites_test.csv", index=False)
eval_units.drop(columns=["seen_text"], errors="ignore").to_csv(OUT / "cascade_decisions_test.csv", index=False)
pd.DataFrame([{"rule": k, "example_original": o, "example_rewrite": r_} for (k, o, r_) in
              [(r.a1_rule, r.text, A1[r.unit_id][0]) for r in tg[(tg.a1_status == "rewritten")].drop_duplicates("a1_rule").itertuples()]]).to_csv(OUT / "rule_templates.csv", index=False)
json.dump({"n_test_ads": 400, "n_target_units_test": int(len(eval_units)), "n_no_rewrite_listings_test": int(len(nr_units)),
           "cascade_unit_acceptance": float(ptn), "cascade_fully_neutralised_ads": float(adm.fully_neutralised.mean()),
           "checks_passed": int(vc.passed.sum()), "checks_total": int(len(vc))}, open(OUT / "results_summary.json", "w"), indent=1)

                                                                                   check  passed       detail
                                      test sample equals the Step 7 file and has 400 ads    True             
                                                    no test ad in training or validation    True             
                                       every target span reproduces inside its unit text    True 0 mismatches
                         residual rules flag at least 95% of the original explicit units    True        1.000
                         residual rules flag at least 95% of the original implicit units    True        0.982
                                      no test unit id is a training source of Approach 2    True             
                                    T5 training sources come from training-pool ads only    True             
                                                all target units have a cascade decision    True             
          

## 11b. Worked example: one job input through detection, classification and the three rewrite approaches

This section runs **one real advertisement supplied by the project owner** (not corpus data, not generated) through the same components used above. It adds no new method, so it relies on the sources already cited for those components (sections 3-6). It is a demonstration of behaviour on a single input, not an evaluation: one advertisement cannot give precision, recall or a pass rate.

**Steps.** (1) Parse and segment the advertisement with the **Step 1-3 code**, read directly from `equal-s1-3-data-standardization-preprocessing.ipynb`, so units, components and offsets follow the corpus pipeline. Contact lines are classified by the parser and are not analysed. (2) Detect with three detectors: the Step 7 curated explicit lexicon (with the both-genders listing rule, code G1), the Step 8 gender-coded stem lists, and the saved Step 6 DistilBERT span tagger (`model_best_seed`). (3) Route each unit and the advertisement to `explicit`, `implicit` or `none` with the v0.3 routing (E codes explicit, I codes implicit, G1 and INC not biased on their own). (4) Rewrite as the pipeline is designed: only units with explicit or implicit wording are rewritten, and a listing that names both genders is left unchanged. (5) A **forced trial** then runs all three approaches on every unit that any detector flagged, including the listing, to show what each approach would do and whether it passes the gate. The forced trial is not part of the pipeline.

**Provenance of the detectors.** The Step 6 tagger was trained on v0.2 rule-derived (silver) labels, not human labels, so its output shows agreement with the rules, not validity. Approach 2 is retrained here with the same seeds, data and parameters as Section 5 because the earlier run kept only its predictions on the test texts; the retraining is checked against those cached predictions below.

In [16]:
# DEMO_CELL parse and segment the supplied advertisement with the Step 1-3 code
import hashlib
nb13 = json.load(open("equal-s1-3-data-standardization-preprocessing.ipynb", encoding="utf-8"))
ns13 = {"re": re, "json": json, "hashlib": hashlib, "pd": pd}
exec("".join(nb13["cells"][4]["source"]).split("\nparsed = ads")[0], ns13)
exec("".join(nb13["cells"][6]["source"]), ns13)

DEMO_RAW = """Puregold

Human Resource Assistant

Qualifications:
~Female/Male
~Graduate of BS Psychology, BSBA Human Resource Management, BS ~Behavioral Science or other related courses 
~With at least 1 year experience in human resources functions
~With knowledge in implementing all human resources faucet

For inquiries and referral, visit or contact:
~web.facebook.com/LMDPESOMarinduqueOfficial
~lmdpeso@yahoo.com
~Tel. No.: (042) 332-0266
~Mobile No.: 0912-673-0915
~Capitol Compound, Santol, Boac, Marinduque
"""
parsed_demo = ns13["parse_ad"](DEMO_RAW)
demo_lines = pd.DataFrame([{"line_no": L["line_no"], "component": L["component"], "rule": L["rule"],
                            "text": L["raw"] if L["component"] not in ("contact",) else "(contact line, not analysed)"} for L in parsed_demo["lines"]])
print("parser flags:", sorted(parsed_demo["flags"]) or "none")
print(demo_lines.to_string(index=False))
demo_units = pd.DataFrame(ns13["build_units"](DEMO_RAW, parsed_demo, "DEMO", "EMP-DEMO"))
demo_units["unit_id"] = ["DEMO-u%02d" % i for i in demo_units["unit_index"]]
print(f"\n{len(demo_units)} units")
print(demo_units[["unit_id", "component", "line_no", "text_masked"]].to_string(index=False))

parser flags: none
 line_no            component             rule                                                                                                        text
       1        employer_name       FIRST_LINE                                                                                                    Puregold
       3            job_title       TITLE_LINE                                                                                    Human Resource Assistant
       5 qualification_header      QUAL_HEADER                                                                                             Qualifications:
       6        qualification    IN_QUAL_BLOCK                                                                                                ~Female/Male
       7        qualification    IN_QUAL_BLOCK ~Graduate of BS Psychology, BSBA Human Resource Management, BS ~Behavioral Science or other related courses
       8        qualification    IN_QUAL_BLOCK     

In [17]:
# DEMO_CELL three detectors on every unit
S6DIR = Path("outputs") / "s6_span_detector" / "model_best_seed"
from transformers import AutoModelForTokenClassification
tok6 = AutoTokenizer.from_pretrained(str(S6DIR)); m6 = AutoModelForTokenClassification.from_pretrained(str(S6DIR)); m6.eval()
LAB6 = [m6.config.id2label[i] for i in range(len(m6.config.id2label))]

def s7_spans(text):
    hits = set(s7_detect(text)); out = []
    for m in TOKRX.finditer(text):
        l = m.group(0).lower()
        if l in hits:
            out.append((m.start(), m.end(), "E1" if l in S7_TERMS else ("E3" if l in S7_PRON else "E2"), m.group(0)))
    return out

def g1_spans(text):
    if not is_listing(text):
        return []
    return sorted({(m.start(), m.end(), "G1", m.group(0)) for rx in (MALE_RX, FEM_RX) for m in rx.finditer(text)})

def s8_spans(text):
    out = []
    for m in CODED_RX.finditer(text):
        w = m.group(0).lower()
        if w.startswith("confidential"):
            continue
        out.append((m.start(), m.end(), "I1" if w.startswith(MASC) else "I2", m.group(0)))
    return out

def s6_spans(text):
    enc = tok6(text, truncation=True, max_length=96, return_offsets_mapping=True, return_tensors="pt")
    offs = enc.pop("offset_mapping")[0].tolist()
    with torch.no_grad():
        pr = m6(**enc).logits.argmax(-1)[0].tolist()
    out, cur = [], None
    for (s, e), p in zip(offs, pr):
        if s == e:
            continue
        tag = LAB6[p]
        if tag == "O":
            if cur: out.append(cur); cur = None
            continue
        pre, k = tag.split("-")
        if pre == "B" or cur is None or cur[2] != k:
            if cur: out.append(cur)
            cur = [s, e, k]
        else:
            cur[1] = e
    if cur: out.append(cur)
    return [(s, e, k, text[s:e]) for s, e, k in out]

ROUTE_OF = lambda code: "explicit" if code in ("E1", "E2", "E3") else ("implicit" if code in ("I1", "I2") else "none")
def route(spans_):
    r = {ROUTE_OF(c) for (_, _, c, _) in spans_}
    return "explicit" if "explicit" in r else ("implicit" if "implicit" in r else "none")

rows = []
for r in demo_units.itertuples():
    t = r.text_masked
    d7, d1, d8, d6 = s7_spans(t), g1_spans(t), s8_spans(t), s6_spans(t)
    rows.append({"unit_id": r.unit_id, "component": r.component, "text": t, "s7_spans": d7, "g1_spans": d1, "s8_spans": d8, "s6_spans": d6})
det = pd.DataFrame(rows)
fmt = lambda sp: "; ".join(f"{w}={c}" for (_, _, c, w) in sp) or "-"
view = pd.DataFrame({"unit": det.unit_id, "text": det.text.str.slice(0, 60), "S7 lexicon": det.s7_spans.map(fmt), "listing rule": det.g1_spans.map(fmt),
                     "S8 stems": det.s8_spans.map(fmt), "S6 tagger": det.s6_spans.map(fmt)})
print(view.to_string(index=False))

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 102/102 [00:00<00:00, 2519.34it/s]

    unit                                                         text S7 lexicon       listing rule S8 stems       S6 tagger
DEMO-u01                                     Human Resource Assistant          -                  -        -               -
DEMO-u02                                                  Female/Male          - Female=G1; Male=G1        - Female/Male=INC
DEMO-u03 Graduate of BS Psychology, BSBA Human Resource Management, B          -                  -        -               -
DEMO-u04 With at least 1 year experience in human resources functions          -                  -        -               -
DEMO-u05    With knowledge in implementing all human resources faucet          -                  -        -               -


In [18]:
# DEMO_CELL classification: unit level and advertisement level, per detector
def rule_union(r):
    return r.s7_spans + r.s8_spans + r.g1_spans
det["route_rules"] = det.apply(lambda r: route(rule_union(r)), axis=1)
det["route_s6"] = det.s6_spans.map(route)
det["code_rules"] = det.apply(lambda r: "+".join(dict.fromkeys(c for (_, _, c, _) in rule_union(r))) or "-", axis=1)
det["code_s6"] = det.s6_spans.map(lambda sp: "+".join(dict.fromkeys(c for (_, _, c, _) in sp)) or "-")
print(det[["unit_id", "component", "code_rules", "route_rules", "code_s6", "route_s6"]].to_string(index=False))
ad_route_rules = route([s for r in det.itertuples() for s in rule_union(r)]); ad_route_s6 = route([s for r in det.itertuples() for s in r.s6_spans])
print(f"\nAdvertisement route (rules: Step 7 lexicon + listing rule + Step 8 stems): {ad_route_rules}")
print(f"Advertisement route (Step 6 tagger): {ad_route_s6}")
print("G1 (both genders listed) is not biased on its own in taxonomy v0.3, so a listing routes to 'none'.")

 unit_id     component code_rules route_rules code_s6 route_s6
DEMO-u01     job_title          -        none       -     none
DEMO-u02 qualification         G1        none     INC     none
DEMO-u03 qualification          -        none       -     none
DEMO-u04 qualification          -        none       -     none
DEMO-u05 qualification          -        none       -     none

Advertisement route (rules: Step 7 lexicon + listing rule + Step 8 stems): none
Advertisement route (Step 6 tagger): none
G1 (both genders listed) is not biased on its own in taxonomy v0.3, so a listing routes to 'none'.


In [19]:
# DEMO_CELL designed pipeline, then a forced trial of the three approaches
def dedup(sp):
    seen, out = set(), []
    for s in sorted(sp):
        if (s[0], s[1]) not in seen:
            seen.add((s[0], s[1])); out.append(s)
    return out

def trial_target(r):
    exp = dedup([s for s in r.s7_spans + r.g1_spans + r.s6_spans if ROUTE_OF(s[2]) == "explicit" or s[2] == "G1"])
    imp = dedup([s for s in r.s8_spans + r.s6_spans if ROUTE_OF(s[2]) == "implicit"])
    return {"exp": [(s, e, "E1" if c == "G1" else c, w) for (s, e, c, w) in exp], "imp": imp}

pipe_tgt = {r.unit_id: {"exp": [s for s in r.s7_spans], "imp": [s for s in r.s8_spans]} for r in det.itertuples()}
designed = []
for r in det.itertuples():
    t = pipe_tgt[r.unit_id]
    if r.g1_spans and not t["exp"]:
        designed.append((r.unit_id, "listing of both genders (G1)", "no_rewrite", r.text))
    elif not t["exp"] and not t["imp"]:
        designed.append((r.unit_id, "no target wording", "unchanged", r.text))
    else:
        designed.append((r.unit_id, "target", "needs cascade", r.text))
designed = pd.DataFrame(designed, columns=["unit_id", "reason", "decision", "text"])
print("Designed pipeline (targets = Step 7 explicit spans and Step 8 stems, as in sections 2 and 7):")
print(designed.to_string(index=False))

trial = [r for r in det.itertuples() if trial_target(r)["exp"] or trial_target(r)["imp"]]
print(f"\nForced trial: {len(trial)} unit(s) flagged by at least one detector")
trial_texts = [r.text for r in trial]
chk_texts = sorted(set(eval_units["text"]))[:12]
DEMO_T5_FILE = OUT / "demo_t5_predictions.json"
if DEMO_T5_FILE.exists():
    DEMO_T5 = json.load(open(DEMO_T5_FILE, encoding="utf-8"))
elif os.environ.get("S9_DEMO_SKIP_T5") == "1":
    DEMO_T5 = {"preds": {s: {t: None for t in trial_texts + chk_texts} for s in seeds}}
else:
    DEMO_T5 = {"preds": {}}
    _saved_eval_texts = eval_texts
    eval_texts = trial_texts + chk_texts
    try:
        for sd in T5C["seeds"]:
            p, _ = train_t5(sd); DEMO_T5["preds"][str(sd)] = p
    finally:
        eval_texts = _saved_eval_texts
    json.dump(DEMO_T5, open(DEMO_T5_FILE, "w", encoding="utf-8"))
repro = {s: np.mean([DEMO_T5["preds"][s][t] == T5_OUT["preds"][s][t] for t in chk_texts]) for s in seeds}
print("retrained Approach 2 reproduces the cached predictions on 12 test texts, per seed:", {k: round(float(v), 3) for k, v in repro.items()})

DEMO_LLM_FILE = OUT / "demo_llm_outputs.json"
demo_prompt = lambda r: f"Line: {r.text}\nIdentified wording to remove or neutralise: {', '.join(dict.fromkeys(w for (_, _, _, w) in trial_target(r)['exp'] + trial_target(r)['imp']))}"
DEMO_LLM = json.load(open(DEMO_LLM_FILE, encoding="utf-8")) if DEMO_LLM_FILE.exists() else {}
need_llm = [demo_prompt(r) for r in trial if demo_prompt(r) not in DEMO_LLM]
if need_llm:
    QWEN_DIR = str(next((Path.home() / ".cache" / "huggingface" / "hub" / "models--Qwen--Qwen2.5-1.5B-Instruct" / "snapshots").iterdir()))
    tokq = AutoTokenizer.from_pretrained(QWEN_DIR); mq = AutoModelForCausalLM.from_pretrained(QWEN_DIR, dtype=torch.float32); mq.eval()
    for k in need_llm:
        msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": k}]
        ids = tokq.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True)
        with torch.no_grad():
            o = mq.generate(**ids, max_new_tokens=LC["max_new_tokens"], do_sample=False, pad_token_id=tokq.eos_token_id)
        DEMO_LLM[k] = tokq.decode(o[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    json.dump(DEMO_LLM, open(DEMO_LLM_FILE, "w", encoding="utf-8"))

def first_fail_demo(g):
    if g["pass"]: return "pass"
    if not g["changed"]: return "unchanged"
    if not g["no_residual"]: return "bias wording remains"
    if not (g["retention"] >= PARAMS["gate"]["content_retention_min"]): return "content dropped"
    if not g["numbers_ok"]: return "number changed"
    if g["content_left"] and not (g["cosine"] >= PARAMS["gate"]["cosine_min"]): return "similarity below threshold"
    return "fluency proxy"

trial_rows = []
for r in trial:
    tgt = trial_target(r)
    a1, rule1, st1 = rule_rewrite(r.text, tgt["exp"], r.component)
    cands = [("A1 rules", a1 if st1 == "rewritten" else None, f"{rule1} ({st1})")]
    for s in seeds:
        cands.append((f"A2 t5-small seed {s}", DEMO_T5["preds"][s].get(r.text), ""))
    cands.append(("A3 Qwen2.5-1.5B", DEMO_LLM[demo_prompt(r)], ""))
    for name, rew, note in cands:
        g = gate(r.text, rew, tgt)
        trial_rows.append({"unit_id": r.unit_id, "original": r.text, "approach": name, "rewrite": rew if rew is not None else "(none)", "note": note,
                           "gate": first_fail_demo(g), "cosine": None if pd.isna(g["cosine"]) else round(g["cosine"], 3)})
trial_df = pd.DataFrame(trial_rows)
trial_df.to_csv(OUT / "demo_single_ad_trial.csv", index=False)
print("\nForced trial (all three approaches on every flagged unit; the listing is a no_rewrite in the pipeline):")
print(trial_df.to_string(index=False))

# the advertisement as the designed pipeline leaves it
final_demo = {r.unit_id: r.text for r in det.itertuples()}
print("\nAdvertisement after the designed pipeline (non-contact units; unchanged units shown as-is):")
for r in demo_units.itertuples():
    mark = "  " if final_demo[r.unit_id] == r.text_masked else "->"
    print(f"{mark} [{r.component}] {final_demo[r.unit_id]}")
det.drop(columns=["s7_spans", "g1_spans", "s8_spans", "s6_spans"]).to_csv(OUT / "demo_single_ad_detection.csv", index=False)

Designed pipeline (targets = Step 7 explicit spans and Step 8 stems, as in sections 2 and 7):
 unit_id                       reason   decision                                                                                                       text
DEMO-u01            no target wording  unchanged                                                                                   Human Resource Assistant
DEMO-u02 listing of both genders (G1) no_rewrite                                                                                                Female/Male
DEMO-u03            no target wording  unchanged Graduate of BS Psychology, BSBA Human Resource Management, BS ~Behavioral Science or other related courses
DEMO-u04            no target wording  unchanged                                               With at least 1 year experience in human resources functions
DEMO-u05            no target wording  unchanged                                                  With knowledge in implementi

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Loading weights:  34%|███▍      | 45/131 [00:00<00:00, 420.36it/s]

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 678.38it/s]

C:\Users\cyber\AppData\Local\Temp\ipykernel_42320\2875275451.py:22: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:821.)
  loss = model(**x, labels=y).loss; loss.backward(); opt.step(); opt.zero_grad(); tot += float(loss) * len(b)


...

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 918.38it/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Loading weights:  24%|██▍       | 32/131 [00:00<00:00, 236.73it/s]

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 887.72it/s]

retrained Approach 2 reproduces the cached predictions on 12 test texts, per seed: {'1': 1.0, '2': 1.0, '3': 1.0}

Forced trial (all three approaches on every flagged unit; the listing is a no_rewrite in the pipeline):
 unit_id    original           approach                          rewrite                    note      gate cosine
DEMO-u02 Female/Male           A1 rules                           (none) G1-listing (no_rewrite) unchanged   None
DEMO-u02 Female/Male A2 t5-small seed 1                      Female/Male                         unchanged   None
DEMO-u02 Female/Male A2 t5-small seed 2                      Female/Male                         unchanged   None
DEMO-u02 Female/Male A2 t5-small seed 3                      Female/Male                         unchanged   None
DEMO-u02 Female/Male    A3 Qwen2.5-1.5B Open to all qualified candidates                              pass   None

Advertisement after the designed pipeline (non-contact units; unchanged units shown as-is):
   [

In [20]:
# DEMO_CELL checks for the worked example
dchk = []
dadd = lambda name, ok, detail="": dchk.append({"check": name, "passed": bool(ok), "detail": detail})
dadd("unit text reproduces from the raw advertisement when nothing was masked",
     all(DEMO_RAW[r.char_start:r.char_end] == r.text_masked for r in demo_units.itertuples() if json.loads(r.mask_edits) == []))
dadd("no contact line became a unit", not set(demo_units.component) & {"contact", "employer_name", "qualification_header"})
dadd("Approach 2 retrained with the same seeds reproduces the cached test predictions (12 texts x 3 seeds)", all(v == 1.0 for v in repro.values()) or os.environ.get("S9_DEMO_SKIP_T5") == "1", str({k: round(float(v), 3) for k, v in repro.items()}))
dadd("every flagged unit has all five candidates in the trial table", all((trial_df.unit_id == r.unit_id).sum() == 2 + len(seeds) for r in trial))
dadd("the designed pipeline leaves a both-genders listing unchanged", all(final_demo[r.unit_id] == r.text for r in det.itertuples() if r.g1_spans and not pipe_tgt[r.unit_id]["exp"]))
dchk = pd.DataFrame(dchk); print(dchk.to_string(index=False)); dchk.to_csv(OUT / "demo_single_ad_checks.csv", index=False)
assert dchk["passed"].all(), "worked-example checks failed"

                                                                                               check  passed                         detail
                             unit text reproduces from the raw advertisement when nothing was masked    True                               
                                                                       no contact line became a unit    True                               
Approach 2 retrained with the same seeds reproduces the cached test predictions (12 texts x 3 seeds)    True {'1': 1.0, '2': 1.0, '3': 1.0}
                                       every flagged unit has all five candidates in the trial table    True                               
                                       the designed pipeline leaves a both-genders listing unchanged    True                               


### 11b.1 Reading the worked example

**Detection and classification.** Of the five units (one title, four qualification lines), the rule detectors flagged one: "Female/Male", which the listing rule codes **G1** (both genders named, no stated preference). Neither the Step 7 lexicon nor the Step 8 stem lists flagged anything else; "Human" in the title is not treated as an "-man" word because it is on the Step 7 stop list. The advertisement therefore routes to **none**, which is the v0.3 route for a nonrestrictive listing. The Step 6 tagger also found no explicit or implicit span, but it labelled "Female/Male" as **INC** because it was trained on v0.2 silver labels, where listings were coded INC. Both detectors give route `none`; the code differs (G1 vs INC) because of the v0.2 to v0.3 change. This is a single input, so no accuracy claim follows.

**Rewriting as designed.** The pipeline has no target unit in this advertisement, so every unit is left unchanged and the listing is a terminal `no_rewrite`. Nothing is sent to the cascade.

**Forced trial.** When the three approaches are forced onto the listing, they differ: Approach 1 declines by rule, Approach 2 copies the line unchanged for all three seeds (as on implicit units above, it only learned to change lines with a single gender word), and Approach 3 replaces it with "Open to all qualified candidates", which the gate accepts. The gate acceptance is only an automatic check (no residual gender word, no number change, no content words to lose). It does not make the rewrite necessary: the original line already includes both genders, and replacing it would remove information the employer stated. We therefore treat the designed `no_rewrite` as the correct outcome for this input.

**Limits of the example.** One advertisement, one unit flagged, no human judgement, a 1.5B model, and detectors whose labels come from rules. Other wording in this advertisement ("Human Resource Assistant", "implementing ... faucet" [sic]) was not flagged, so the example does not test the implicit path. The contact lines were parsed as contact and not analysed.

## 12. Interpretation and limitations

All figures are on the 400-ad test sample (249 rewrite-target units: 94 explicit, 153 implicit, 2 both). "Gate pass" means the rewrite passed the automatic checks G1–G6. It is **not** a human judgement of neutrality, fluency or meaning. Intervals are 95% bootstrap intervals over ads.

### 12.1 Interpretation

**Explicit bias is almost fully rewritable by rules.** Approach 1 passed the gate on 0.968 of explicit units (0.928–1.000). Approach 2 reached 0.947 and Approach 3 0.957, and the differences between them include 0 (A3 minus A2: +0.011, −0.033 to +0.062). We found that explicit gender wording in this corpus is short and formulaic: 86 of 94 explicit test units are texts that were already training sources, mostly the bare "Male" or "Female". The three approaches therefore agree because the task is close to a lookup. The proposed rule templates are consistent with the template-based and rule-based approaches reported for gendered-language correction [3], [4], but this sample cannot show that they would cope with unseen, syntactically embedded phrasing: only 8 explicit test units have unseen text (gate pass A1 0.625, A2 0.375, A3 0.625).

**Implicit bias is where the approaches separate.** A1 has no implicit templates, by design. A2 changed 0.7% of implicit units, so it behaves as a copy model: it was trained on only 28 changed pairs, all explicit. Only A3 produced implicit rewrites, with a gate pass of 0.373 (0.292–0.449) and an A3 minus A2 difference of +0.373 (0.290–0.455, excludes 0). Our study suggests that the ability to rewrite implicit wording comes from the instruction-following model and not from supervised training on silver pairs. This points the same way as the reports that LLM-guided rewriting handles bias that templates cannot [5], [7], [8], but our model is a 1.5B local model and not the GPT-4 class used in that literature, so the size of the effect is not comparable.

**The cascade adds little beyond A3 alone.** The cascade accepted 149 of 249 units (0.598, 0.532–0.664) against 0.590 for A3. The gain is +0.008 (−0.008 to +0.024). Its advantage is not acceptance, it is control: explicit units are decided by deterministic rules (91 by A1), so the LLM touches only 1 explicit unit (and 57 implicit). The proposed cascade may benefit from this routing without adversely impacting acceptance.

**Semantic preservation.** Median embedding cosine for changed rewrites was 0.82–0.90 and median content retention for A3 implicit rewrites was 0.894. The gate pass rate for A3 on implicit units is sensitive to the cosine threshold (0.386 at τ=0.5, 0.373 at 0.7, 0.203 at 0.9), so the headline 0.373 depends on our own choice of 0.70. The failure analysis shows what blocked A3 on implicit units: residual coded wording (46), dropped content (36), similarity below threshold (3), unchanged (11). Almost half of A3's attempts therefore either left the coded word in place or removed too much.

**What the gate cannot see.** 27 of the 57 accepted implicit rewrites are deletion-only. Examples: "Ability to analyze and solve problems" → "Ability to solve problems"; "Excellent leadership and motivational skills" → "Excellent motivational skills"; "Has good positive attitude and trustworthy" → "Has good positive attitude". The gate excludes flagged words when it checks content retention, so it accepts the removal of a *requirement* as if it were the removal of *bias*. Many implicit "coded" words are legitimate requirements ("analytical", "leadership", "support"). The reported implicit gate pass of 0.373 is therefore an upper bound on faithful rewriting. Substitutions such as "confidentiality → privacy" and "Analyst → Specialist" also change meaning or are our own design choices, with no verified source.

**Ad level.** 101 of 183 ads with at least one rewrite target (0.552) were fully neutralised. Explicit-only ads were fully neutralised in 0.986 of cases (73 of 74; the exception holds the pregnancy or height line sent to manual review), while ads with an implicit unit were fully neutralised in only 0.20–0.27. A heuristic Flesch score changed by about 0.0 on the median ad. This is a weak proxy and is not evidence about readability.

**Consistency with the literature, and gaps.** The pattern (rules sufficient for explicit cases, LLMs needed for implicit ones, and a risk of meaning loss) agrees with [1]–[9] in direction only. We found no verified 2022–2025 source for: our cosine and retention thresholds, training a seq2seq model on rule-derived pairs, Flesch on job-ad fragments, or bootstrapping over ads. These are our own design choices and are stated as such.

### 12.2 Limitations and their likely impact

1. **The gate is circular.** G2 reuses the word lists that produced the silver labels, so it tests consistency, not validity. Impact: bias-removal figures are optimistic; a reader may still perceive bias in a rewrite the gate accepts.
2. **No human evaluation (by design for this stage).** Readability, perceived neutrality, content preservation and fluency were not rated by people. Impact: every quality claim here rests on proxies; the 7-point Likert protocol from the brief is not run.
3. **Dropped requirements pass the gate.** 27 of 57 accepted implicit rewrites only delete words. Impact: true faithful implicit rewriting is lower than 0.373, perhaps near 0.20 (30 of 153 units) if deletions are counted as failures.
4. **Approach 2 is a student of the rules.** It trained on 28 changed pairs and 239 identity pairs, with no implicit pairs, and its test explicit texts are mostly seen. Impact: its explicit score reflects memorisation and says nothing about generalisation; its implicit score of 0 reflects the lack of training data and not the architecture.
5. **A 1.5B local model replaces GPT-4.** Impact: Approach 3 results are not comparable to GPT-4 results in the literature; a stronger model would likely reduce residual wording and dropped content.
6. **Thresholds are our own.** Cosine ≥ 0.70 and retention ≥ 0.80 have no verified source. Impact: the implicit pass rate moves from 0.386 to 0.203 across τ=0.5–0.9.
7. **Silver labels and employer overlap.** Targets come from the lexicon, and 362 of 400 test ads share an employer with the pool. Impact: positive results can be partly employer templates.
8. **Role-noun implicit spans.** Excluding the 19 role-noun-only units lowers A3 implicit pass from 0.373 to 0.351. These titles ("Analyst" → "Specialist") are judgement calls.
9. **"Both" units fail entirely.** The 2 units naming both genders with other bias are all sent to review. Impact: small count, so no estimate is possible.
10. **Manual-review rule added after seeing outputs.** Pregnancy and sex-conditioned height requirements stop at review (2 test units). This was chosen after seeing that A3 had rewritten a pregnancy line into an unreasonable one. Impact: the explicit acceptance of 0.979 excludes them by construction.
11. **Single-unit input.** Units are rewritten without the surrounding ad. Impact: lines that depend on context (for example "Foreman" in a title and in experience text) may be rewritten inconsistently.
12. **English-oriented models with Filipino and typo-laden text.** Examples include "Trsutworthy". Impact: cosine and the stem lists may under-handle such text.
13. **Not measured:** actual readability, grammaticality, and perceived neutrality.

## 13. References (IEEE)

Only sources verified in `literature-support-for-methods.md` are used for claims; reading depth is stated there. Items marked as outside the 2022-2025 window are cited only to identify a resource.

[1] S. Hu *et al.*, "Balancing gender bias in job advertisements with text-level bias mitigation," *Front. Big Data*, vol. 5, Art. no. 805713, 2022, doi: 10.3389/fdata.2022.805713.

[2] J. C. He and S. K. Kang, "Debiasing job ads by replacing masculine language increases gender diversity of applicant pools," *Proc. Natl. Acad. Sci. U.S.A.*, vol. 122, no. 7, Art. no. e2409854122, 2025, doi: 10.1073/pnas.2409854122 (the publisher page returned an error when checked; confirm details before submission).

[3] C. Amrhein, F. Schottmann, R. Sennrich, and S. Läubli, "Exploiting biased models to de-bias text: A gender-fair rewriting model," in *Proc. 61st Annu. Meeting Assoc. Comput. Linguistics (ACL)*, vol. 1, 2023, pp. 4486-4506, doi: 10.18653/v1/2023.acl-long.246.

[4] P. Mainardi, F. Garcea, and A. Barrón-Cedeño, "Fine-tuning vs prompting techniques for gender-fair rewriting of machine translations," in *Proc. 6th Workshop Gender Bias Natural Lang. Process. (GeBNLP)*, 2025, pp. 320-337, doi: 10.18653/v1/2025.gebnlp-1.28.

[5] L. Shu *et al.*, "RewriteLM: An instruction-tuned large language model for text rewriting," *Proc. AAAI Conf. Artif. Intell.*, vol. 38, no. 17, pp. 18970-18980, 2024, doi: 10.1609/aaai.v38i17.29863.

[6] I. O. Gallegos *et al.*, "Self-debiasing large language models: Zero-shot recognition and reduction of stereotypes," in *Proc. NAACL (Short Papers)*, 2025, pp. 873-888, doi: 10.18653/v1/2025.naacl-short.74 (abstract read).

[7] C. Davis *et al.*, "Prompting open-source and commercial language models for grammatical error correction of English learner text," in *Findings ACL 2024*, 2024, pp. 11952-11967, doi: 10.18653/v1/2024.findings-acl.711 (abstract read).

[8] M. T. I. Khondaker, M. Abdul-Mageed, and L. V. S. Lakshmanan, "DetoxLLM: A framework for detoxification with explanations," in *Proc. EMNLP*, 2024, pp. 19112-19139, doi: 10.18653/v1/2024.emnlp-main.1066 (abstract read).

[9] D. Kumar, S. Masoudian, A. B. Melchiorre, and M. Schedl, "Mitigating gender bias in job ranking systems using job advertisement neutrality," in *Proc. 4th Workshop NLP for Positive Impact (NLP4PI)*, 2025, pp. 264-271, doi: 10.18653/v1/2025.nlp4pi-1.23.

[10] R. Blackwell *et al.*, "Towards reproducible LLM evaluation: Quantifying uncertainty in LLM benchmark scores," arXiv:2410.03492, 2024 (preprint).

[11] F. Feng, Y. Yang, D. Cer, N. Arivazhagan, and W. Wang, "Language-agnostic BERT sentence embedding," in *Proc. 60th Annu. Meeting Assoc. Comput. Linguistics (Vol. 1: Long Papers)*, 2022, pp. 878-891, doi: 10.18653/v1/2022.acl-long.62 (background for a multilingual encoder; the encoder used here is `paraphrase-multilingual-MiniLM-L12-v2`, not LaBSE).

*Outside the 2022-2025 window:* N. Reimers and I. Gurevych, "Sentence-BERT: Sentence embeddings using Siamese BERT-networks," in *Proc. EMNLP-IJCNLP*, 2019, doi: 10.18653/v1/d19-1410 (basis of the sentence-embedding framework); C. Raffel *et al.*, "Exploring the limits of transfer learning with a unified text-to-text transformer," *J. Mach. Learn. Res.*, 2020 (T5; not verified in this session).